# GCP Pose Estimation — Kaggle Training

## Before running: download the data as a ZIP and upload to Kaggle

**1. Download the Drive folder as ZIP (on your laptop)**
- Open [drive.google.com](https://drive.google.com) → find the shared GCP data folder
- Right-click the folder → **Download**
- Chrome will zip it and download. If the folder is >2 GB, Drive splits it into multiple ZIPs — download **all** of them.

**2. Upload the ZIP(s) as a Kaggle dataset**
- kaggle.com → your profile → **Datasets** → **New Dataset**
- Upload all ZIP file(s) → title: `skylark-gcp-data` → visibility: **Private** → Create

**3. Add that dataset as input to this notebook**
- In this notebook → **Add Input** (top right) → Your Datasets → `skylark-gcp-data` → Add

Then run cells top to bottom. GPU T4×2 + Internet must both be ON.

In [1]:
# 1. Clone repo + install dependencies
!git clone https://github.com/J4KE-B/skylark-gcp gcp
%cd gcp
!pip -q install -r requirements.txt

Cloning into 'gcp'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 113 (delta 44), reused 101 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 38.61 KiB | 5.52 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/kaggle/working/gcp
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 111.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you 

In [2]:
# 2. Wire config to Kaggle input dataset
import os, json, yaml, glob
from pathlib import Path

hits = sorted(glob.glob('/kaggle/input/**/gcp_marks.json', recursive=True), key=len)
assert hits, 'gcp_marks.json not found — add skylark-gcp-data as notebook input'
label_path = hits[0]
BASE = os.path.dirname(label_path)
print('Dataset root:', BASE)
print('Contents:', sorted(os.listdir(BASE)))

# Load + patch labels first; we use a real label key to locate the image root
os.makedirs('data', exist_ok=True)
raw = json.load(open(label_path))
patched = {k.replace('&', 'and'): v for k, v in raw.items()}
local_labels = 'data/gcp_marks.json'
json.dump(patched, open(local_labels, 'w'))
n_fixed = sum(1 for k in raw if '&' in k)
if n_fixed: print(f'Patched {n_fixed} label keys (& -> and)')

def find_dir(base, *names):
    for n in names:
        p = os.path.join(base, n)
        if os.path.isdir(p): return p
    raise FileNotFoundError(f'None of {names} in {base}')

def resolve_by_label(base, label_keys):
    """Find the dir that actually holds the labelled site folders, regardless of wrapper
    levels (train_clean/train/train_dataset/...) or junk dirs (__MACOSX). Locates the site
    folder of a sample label key and returns its parent."""
    site = next(iter(label_keys)).split('/')[0]
    for p in Path(base).rglob(site):
        if p.is_dir() and '__MACOSX' not in p.parts:
            return str(p.parent)
    return base

def descend_to_images(base, max_depth=8):
    """Test set (no labels): descend, skipping junk, until a folder holds images or branches."""
    junk = {'__MACOSX', '.ipynb_checkpoints'}
    d = base
    for _ in range(max_depth):
        entries = [e for e in os.listdir(d) if e not in junk]
        subs = [e for e in entries if os.path.isdir(os.path.join(d, e))]
        has_img = any(f.lower().endswith(('.jpg', '.jpeg')) for f in entries)
        if has_img or len(subs) != 1:
            return d
        d = os.path.join(d, subs[0])
    return d

train_base = find_dir(BASE, 'train_clean', 'train_dataset', 'train')
test_base  = find_dir(BASE, 'test_clean',  'test_dataset',  'test')
train_dir  = resolve_by_label(train_base, patched)
test_dir   = descend_to_images(test_base)
print(f'train_dir : {train_dir}')
print(f'test_dir  : {test_dir}')

cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['paths'] = {'label_file': local_labels, 'train_dir': train_dir,
                'test_dir': test_dir, 'output_dir': 'outputs'}
yaml.safe_dump(cfg, open('configs/config.yaml', 'w'))
print('Config updated.')

labels = json.load(open(local_labels))
found  = sum(1 for k in labels if os.path.exists(os.path.join(train_dir, k)))
pct    = 100 * found // len(labels)
print(f'Train images on disk: {found}/{len(labels)} ({pct}%)')
print('OK — enough to train.' if found >= len(labels) * 0.8 else
      'WARNING: <80% images — check folder names above.')

Dataset root: /kaggle/input/datasets/jacobantonyjeejo/skylark-gcp-data
Contents: ['gcp_marks.json', 'test_clean', 'train_clean']
train_dir : /kaggle/input/datasets/jacobantonyjeejo/skylark-gcp-data/train_clean/train/train_dataset
test_dir  : /kaggle/input/datasets/jacobantonyjeejo/skylark-gcp-data/test_clean/test_dataset
Config updated.
Train images on disk: 1000/1000 (100%)
OK — enough to train.


In [3]:
# 3. SMOKE: 3 epochs, then inspect predictions schema before committing GPU time
import yaml
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 3
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import json; p = json.load(open('predictions.json'))
k = next(iter(p)); print(k, '->', p[k]); print('total:', len(p))

device: cuda
/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 170MB/s]
Ep  1 val loss=1.027 PCK10=0.000 PCK25=0.000 F1=0.929
  -> saved best (score=0.949)
Ep  2 val loss=0.936 PCK10=0.007 PCK25=0.033 F1=0.953
  -> saved best (score=1.190)
Ep  3 val loss=0.796 PCK10=0.053 PCK25=0.184 F1=0.953
  -> saved best (score=1.466)
/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
infer: 100%|████████████████████████████████████| 25/25 [00:33<00:00,  1.33s/it]
wrote 300 predictions -> predictions.json
231129_CTD/231129_CTD_GDA94/11/DJI_20231129142314_01

In [4]:
# 4. FULL run: 60 epochs (early-stops on patience=15)
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 60
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml

device: cuda
/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
Ep  1 val loss=1.011 PCK10=0.000 PCK25=0.000 F1=0.964
  -> saved best (score=0.970)
Ep  2 val loss=0.889 PCK10=0.020 PCK25=0.086 F1=0.942
  -> saved best (score=1.212)
Ep  3 val loss=0.800 PCK10=0.033 PCK25=0.132 F1=0.988
  -> saved best (score=1.337)
Ep  4 val loss=0.912 PCK10=0.046 PCK25=0.158 F1=0.947
  -> saved best (score=1.349)
Ep  5 val loss=0.766 PCK10=0.053 PCK25=0.243 F1=0.990
  -> saved best (score=1.615)
Ep  6 val loss=0.783 PCK10=0.072 PCK25=0.289 F1=0.957
  -> saved best (score=1.661)
Ep  7 val loss=0.914 PCK10=0.105 PCK25=0.270 F1=0.964
Ep  8 val loss=0.802 PCK10=0.053 PCK25=0.283 F1=0.977
Ep  9 val loss=0.872 PCK10=0.132 PCK25=0.303 F1=0.937
Ep 10 val loss=0.869 PCK10=0.092 PCK25=0.250 F1=0.953
Ep 11 val loss=1.126 PCK10=0.079 PCK25=0.237 F1=0.953
Ep 12 val loss=0.965 PCK10=0.086 PCK25=0

In [5]:
# 5. Final predictions + download artifacts
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import pandas as pd; pd.read_csv('outputs/training_log.csv').tail()

/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
infer: 100%|████████████████████████████████████| 25/25 [00:30<00:00,  1.21s/it]
wrote 300 predictions -> predictions.json


,epoch,train_loss,train_kp,train_cls,train_pck10,train_pck25,train_pck50,train_f1,val_loss,val_kp,val_cls,val_pck10,val_pck25,val_pck50,val_f1
49,50,0.496044,0.493643,0.004801,0.172170,0.652123,0.825472,0.998545,1.006483,0.926359,0.160248,0.164474,0.381579,0.506579,0.959064
50,51,0.494126,0.493511,0.001229,0.200472,0.637972,0.831368,1.000000,0.961807,0.905451,0.112711,0.177632,0.388158,0.493421,0.959064
51,52,0.496788,0.496103,0.001371,0.185142,0.614387,0.810142,1.000000,1.001158,0.928058,0.146200,0.111842,0.375000,0.506579,0.964258
52,53,0.494019,0.493600,0.000838,0.174528,0.619104,0.837264,1.000000,1.012067,0.923584,0.176966,0.171053,0.388158,0.493421,0.964100
53,54,0.496135,0.493805,0.004660,0.176887,0.621462,0.831368,0.998545,0.983184,0.916190,0.133988,0.157895,0.355263,0.500000,0.969425


In [6]:
# 6. Visualize predictions at ORIGINAL resolution — hollow circle (marker stays visible)
import os, json, glob, cv2, shutil

pred_hits = sorted(glob.glob('/kaggle/working/**/predictions.json', recursive=True), key=len)
assert pred_hits, 'predictions.json not found — run the predict cell first'
preds = json.load(open(pred_hits[0]))
print(f'Loaded {len(preds)} predictions from {pred_hits[0]}')

def descend_to_images(base, max_depth=8):
    junk = {'__MACOSX', '.ipynb_checkpoints'}
    d = base
    for _ in range(max_depth):
        entries = [e for e in os.listdir(d) if e not in junk]
        subs = [e for e in entries if os.path.isdir(os.path.join(d, e))]
        has_img = any(f.lower().endswith(('.jpg', '.jpeg')) for f in entries)
        if has_img or len(subs) != 1:
            return d
        d = os.path.join(d, subs[0])
    return d

test_base = sorted(glob.glob('/kaggle/input/**/test_clean', recursive=True) +
                   glob.glob('/kaggle/input/**/test_dataset', recursive=True), key=len)[0]
test_dir = descend_to_images(test_base)
print('test_dir:', test_dir)

out_dir = '/kaggle/working/viz'
shutil.rmtree(out_dir, ignore_errors=True); os.makedirs(out_dir)

drawn = 0
for key, val in preds.items():
    p = os.path.join(test_dir, key)
    if not os.path.exists(p): continue
    img = cv2.imread(p)
    if img is None: continue
    h, w = img.shape[:2]
    x, y = int(val['mark']['x']), int(val['mark']['y'])
    s = val['verified_shape']
    r  = max(int(w * 0.03),  70)   # circle radius — encircles the marker so it stays visible
    th = max(int(w * 0.003), 3)    # outline thickness
    fs = max(w / 600.0, 1.5)       # font scale
    cv2.circle(img, (x, y), r, (0, 0, 255), th)            # HOLLOW circle (no fill)
    cv2.putText(img, f'{s} ({x},{y})', (x + r + 8, y - r - 8),
                cv2.FONT_HERSHEY_SIMPLEX, fs, (0, 255, 255), th)
    cv2.imwrite(os.path.join(out_dir, key.replace('/', '_')), img,
                [cv2.IMWRITE_JPEG_QUALITY, 100])
    drawn += 1

shutil.make_archive('/kaggle/working/viz_output', 'zip', out_dir)
sz = os.path.getsize('/kaggle/working/viz_output.zip') / 1e6
print(f'Annotated {drawn} full-res images -> {out_dir}')
print(f'viz_output.zip = {sz:.0f} MB')

Loaded 300 predictions from /kaggle/working/gcp/predictions.json
test_dir: /kaggle/input/datasets/jacobantonyjeejo/skylark-gcp-data/test_clean/test_dataset
Annotated 300 full-res images -> /kaggle/working/viz
viz_output.zip = 2204 MB


## Artifacts to download

From the Kaggle Output panel (right sidebar), download these three files:

- `gcp/outputs/best.pt` — model weights (upload to Drive, copy the shareable link)
- `gcp/outputs/training_log.csv` — open it, read the last row for PCK@10/25/50 + Macro-F1
- `gcp/predictions.json` — this is your submission file for Skylark

Then update `README.md` in the repo:
1. Fill in the Results table with the PCK/F1 numbers
2. Add the Drive link for `best.pt`
3. `git add README.md && git commit -m "docs: add training results and weights link" && git push`